# Genomic Survival Analysis in Clinical Oncology

#### This project analyzes patient survival outcomes and genomic mutation profiles to identify mutation-related mortality risk in cancer cohorts. Data was loaded from three cBioPortal datasets from MSK-IMPACT 2017.

## Key Objectives/Analyses
* Cohort Selection: Filtered to focus on NSCLC ($n=1542$) to evaluate within single tumor histology. Targeted therapies involve specific gene mutations and specific tumor types.
* Multivariate Risk Modeling: Fit a Cox Proportional Hazards model across mutated genes to determine predictors of overall survival. Derived HRs and statistical significance. Created forest plot for visualization.
* Survival Differences: Created Kaplan-Meier curve to visualize significant survival differences across cohorts.
* Discussion about significant genes and targeted therapies in relation to patient mutations.

## Packages
Python (`pandas`, `numpy`, `lifelines`, `matplotlib`, `seaborn`, `requests`)

In [ ]:
# Standard libraries
import io
import tarfile

# Third-party libraries
import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
import numpy as np

### Dataset Merging

Patient-level clinical profiles + clinical samples + mutations from MSK-IMPACT 2017 joined on `PATIENT_ID`

In [ ]:
# Download MSK-IMPACT 2017 study archive from cBioPortal
# URLs for Patient and Sample level data
patient_url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/msk_impact_2017/data_clinical_patient.txt"
sample_url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/msk_impact_2017/data_clinical_sample.txt"

# Load both datasets
df_patient = pd.read_csv(patient_url, sep="\t", comment="#")
df_sample = pd.read_csv(sample_url, sep="\t", comment="#")

# Merge on PATIENT_ID
df_msk = pd.merge(df_patient, df_sample, on="PATIENT_ID")

# URL for Mutation data
archive_url = "https://datahub.assets.cbioportal.org/msk_impact_2017.tar.gz"

response = requests.get(archive_url)

with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:gz") as tar:
    for member in tar.getmembers():
        if member.name.endswith("data_mutations.txt"):
            f = tar.extractfile(member)
            mut_df = pd.read_csv(f, sep="\t", comment="#", low_memory=False)

### Genomic Feature Engineering and Binary Matrix Construction

Identified top 10 most frequently mutated genes and constructed binary mutation matrix for survival modeling.

In [ ]:
# Count one mutation per gene per patient
unique_muts = mut_df[['Tumor_Sample_Barcode', 'Hugo_Symbol']].drop_duplicates()

# Count how many unique patients have a mutation for each gene to find top 10
top_10_genes = unique_muts['Hugo_Symbol'].value_counts().head(10).index.tolist()

# Filter the mutation dataframe to only include rows for these top 10 genes
filtered_muts = mut_df[mut_df['Hugo_Symbol'].isin(top_10_genes)]
print(f"Filtered mutation dataset shape: {filtered_muts.shape}")

# Transform mutation data into one patient per row, one gene per column format (binary mutation matrix)
# (1 = mutated, 0 = wild-type)
# Rows = samples, columns = genes
gene_matrix = pd.crosstab(filtered_muts['Tumor_Sample_Barcode'], filtered_muts['Hugo_Symbol'])
gene_matrix = (gene_matrix > 0).astype(int)

# Add mut prefix to the column names for easy filtering later
gene_matrix.columns = [f"mut_{gene}" for gene in gene_matrix.columns]

# Merge mutation matrix with the combined dataframe from earlier (df_msk)
merge_col = 'SAMPLE_ID'

df_final = df_msk.merge(gene_matrix, left_on=merge_col, right_index=True, how='left')

mut_cols = df_final.filter(regex = '^mut_').columns

# Missing mutation values = 0
df_final[mut_cols] = df_final[mut_cols].fillna(0).astype(int)

### Data filtering and mapping, pointplot visualization

* Isolated NSCLC (Non-small cell lung cancer) as the indication with the most data in the dataset. One indication has been chosen because analyzing mutations across pooled cancer types introduces confounding since a mutation's impact depends on whether certain treatments are approved by the FDA for certain indications.
* 

In [ ]:
# Finding the top 5 cancer types by patient count
top_5_types = df_final['CANCER_TYPE'].value_counts().head(5).index.tolist()
cancer_type_df = df_final[df_final['CANCER_TYPE'].isin(top_5_types)]

# Filter for NSCLC
single_cancer_df = cancer_type_df[cancer_type_df['CANCER_TYPE'] == 'Non-Small Cell Lung Cancer'].copy()

# Melt mutation columns into a single 'Gene' and 'Status' column
#mutations_to_check = ['mut_TP53', 'mut_KRAS', 'mut_EGFR']
mutations_to_check = [col for col in single_cancer_df.columns if col.startswith("mut_")]

### Cox Proportional Hazards Multivariate Regression

Hazard ratios and 95% confidence intervals were estimated across multiple genetic subtypes to evaluate the impact of oncogenic mutations on overall survival.

Note: This model lacks baseline clinical covariates (age, smoking history, performance status) and therefore is not controlled for any non-genetic confounding factors.

In [ ]:
# Isolate data for Cox model
model_df = single_cancer_df[['OS_MONTHS', 'OS_STATUS'] + mutations_to_check].copy()

# Filter out missing values
model_df = model_df.dropna(subset=['OS_MONTHS', 'OS_STATUS'])

# OS_STATUS 1=event occurred, 0=censored/alive
model_df['OS_STATUS']=model_df['OS_STATUS'].map({
    '0:LIVING': 0,
    '1:DECEASED': 1
})

# Fit Cox Proportional Hazards model
cph = CoxPHFitter()
cph.fit(model_df, duration_col='OS_MONTHS', event_col='OS_STATUS')
cph.print_summary()

### Findings and Interpretation

Out of the 10 genes evaluated across this cohort of $N=1,542$ NSCLC patients ($501$ observed OS events), only three genes reached statistical significance (p<0.05)including TP53, EGFR, and KMT2D.

* `TP53` (Adverse Risk): Mutations significantly associated with an increased risk of mortality. Patients with TP53 mutation exhibited a $62\%$ higher hazard rate compared to wild-type controls.
    * $\text{HR} = 1.62$, $95\%\text{ CI: } 1.34 - 1.95$, $p < 0.005$
* `KMT2D` (Adverse Risk): Mutations significantly associated with increased risk of mortality where patients exhibited a $40\%$ increase in mortality risk.
    * $\text{HR} = 1.40$, $95\%\text{ CI: } 1.04 - 1.89$, $p = 0.03$
* `EGFR` (Favorable Risk): Mutations were significantly associated with reduced mortality risk and showed a $40\%$ reduction in hazard rate relative to wild-type controls.
    * $\text{HR} = 0.60$, $95\%\text{ CI: } 0.47 - 0.76$, $p < 0.005$
* Other genes: The remaining genes did not show statistically significant prognostic value in this model.

Model Performance: This model has a Harrell's Concordance Index of $0.59$, which means somewhat weak predictive discrimination.

### Forest Plot (Cox Model HRs)

In order to visually summarize the independent prognostic impact of each gene mutation, hazard ratios and corresponding $95\%$ confidence intervals from the multivariate Cox proportional hazards model are presented in the forest plot below.

Note the reference line at $\text{HR} = 1.0$.

In [ ]:
plt.figure(figsize=(8, 6))

summary = cph.summary.copy()
y_pos = np.arange(len(summary))

for i in range(len(summary)):
    p_val = summary['p'].iloc[i]
    is_sig = summary['p'].iloc[i] < 0.05

    # Set color based on significance
    color = "#2F4CB3" if is_sig else "#555454"

    # Error bars for hazard ratios with 95% confidence intervals
    plt.errorbar(
        x = summary['exp(coef)'].iloc[i],
        y = y_pos[i],
        color = color,
        xerr = [[summary['exp(coef)'].iloc[i] - summary['exp(coef) lower 95%'].iloc[i]],
                [summary['exp(coef) upper 95%'].iloc[i] - summary['exp(coef)'].iloc[i]]],
        fmt = 'o',
        capsize = 3
    )

    # Add labels for HRs
    label_text = f"{summary['exp(coef)'].iloc[i]:.2f} [{summary['exp(coef) lower 95%'].iloc[i]:.2f}, {summary['exp(coef) upper 95%'].iloc[i]:.2f}]"

    # Annotate point with the labels
    plt.text(
        x = summary['exp(coef) upper 95%'].iloc[i] + 0.2,  # Position to the right of the error bar
        y = y_pos[i],
        s = label_text,
        ha = 'center',
        va = 'bottom',
        fontsize = 9,
        color = color
    )

# Get rid of top and right spines for cleaner look
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.axvline(1.0, color='black', linestyle='--', alpha=0.5)
plt.xlabel("Hazard Ratio (HR) [95% CI]")
plt.ylabel("Mutations")
plt.title('Forest Plot of Hazard Ratios for Mutations in NSCLC')
plt.yticks(ticks = range(len(summary)), labels = summary.index)
plt.show()

### Forest Plot Interpretation

The highlighted plots display the previously discussed results from the Cox model summary. TP53 and KMT2D are significant adverse prognostics and EGFR is a significant favorable prognostic. All other genes have confidence interval error bars that span across the reference line and therefore their association with OS are not statistically significant.

### Kaplan-Meier Survival Analysis for Statistically Significant Genes

In order to visually evaluate the survival trajectories of genomic mutations, KM curves were constructed for the three statistically significant mutations and their wild-type counterparts. For each gene, overall survival in months is compared between mutated (blue) and wild-type (orange) patient cohorts. The shaded bands represent $95\%$ confidence intervals around the survival estimates.

In [ ]:
# Check the statistically significant mutations (TP53, EGFR, KMT2D)
gene_col = ['mut_TP53', 'mut_EGFR', 'mut_KMT2D']
kmf = KaplanMeierFitter()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Filter and plot each mutation status for the top 3 significant genes
for i, gene in enumerate(gene_col):
    ax = axes[i]

    # Mutated group
    mutated_mask = model_df[gene] == 1
    kmf.fit(
        durations=model_df.loc[mutated_mask, 'OS_MONTHS'], 
        event_observed=model_df.loc[mutated_mask, 'OS_STATUS'],
        label = f'{gene} Mutated'
    )
    kmf.plot_survival_function(ax=ax, ci_show=True)

    # Wild-type group
    wt_mask = model_df[gene] == 0
    kmf.fit(
        durations=model_df.loc[wt_mask, 'OS_MONTHS'], 
        event_observed=model_df.loc[wt_mask, 'OS_STATUS'],
        label = f'{gene} WT'
    )
    kmf.plot_survival_function(ax=ax, ci_show=True)

    ax.set_title(f'Overall Survival by {gene} Status')
    ax.set_xlabel('Overall Survival (Months)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower left')

    if i == 0:
        ax.set_ylabel('Survival Probability')

plt.show()

### KM Curves Interpretation

* TP53 (Adverse Outcome): Patients with this mutation exhibit consistently lower survival probabilities throughout the follow-up period compared to wild-type controls. The survival curve separates early (around 5 months) and continues until there is small overlap between the confidence intervals (around 30 months).
* EGFR (Favorable Outcome): EGFR-mutated patients demonstrate a clear survival advantage as the mutated curve remains elevated above the wild-type trajectory across all observed time points, with only slight overlap in confidence intervals (around 25 months).
* KMT2D (Adverse Outcome): This mutation reduced overall survival, with the mutated trajectory dropping noticeably below the wild-type curve. The wider confidence interval bands reflect a smaller sub-cohort of *KMT2D*-mutated cases and therefore more overlaps between the bands occur. This survival curve still clearly shows the advantage of wild-type patients.

**Summary:** The Kaplan-Meier analysis validates the multivariate Cox model findings. *TP53* and *KMT2D* mutations serve as significant markers for reduced overall survival in NSCLC, whereas *EGFR* mutations are associated with favorable long-term survival trajectories.

## Conclusion

### Executive Summary
This analysis evaluated the prognostic impact of key oncogenic mutations on overall survival in a cohort of non-small cell lung cancer (NSCLC) patients ($N = 1,542$; $501$ overall survival events) using multivariate Cox proportional hazards modeling and Kaplan-Meier survival analysis. 

Among the 10 target genes evaluated:
* **Adverse Risk Factors:** TP53 ($\text{HR} = 1.62$, $p < 0.005$) and *KMT2D* ($\text{HR} = 1.40$, $p = 0.03$) were identified as independent predictors of significantly reduced overall survival.
* **Favorable Risk Factor:** EGFR mutations ($\text{HR} = 0.60$, $p < 0.005$) demonstrated a statistically significant positive association with survival outcomes.
* **Non-Significant Biomarkers:** Mutational status for the remaining seven genes (including KRAS, PIK3CA, and TERT) did not demonstrate statistically significant prognostic value when evaluated simultaneously ($p \ge 0.05$).

### Analytical Limitations
* **Unadjusted Model Scope:** The current model evaluates genetic mutations in isolation. Baseline clinical covariates known to impact NSCLC mortality—such as tumor stage, patient age, ECOG performance status, smoking history, and systemic treatment regimens were not incorporated.
* **Predictive Discrimination:** The model achieved a Harrell’s Concordance Index (C-index) of $0.59$. While typical for gene-only models, this score highlights that mutational status alone provides modest predictive power for individual patient risk stratification without clinical context.
* **Treatment Confounding:** The favorable survival associated with EGFR mutations likely reflects the efficacy of targeted tyrosine kinase inhibitor (TKI) therapies in EGFR-mutated NSCLC rather than an inherently indolent tumor biology.

### Proposed Next Steps
1. **Clinical Covariate Adjustment:** Extend the multivariate Cox model to incorporate key clinical parameters (stage, age, sex, pack-years, prior treatments) to calculate fully adjusted hazard ratios ($\text{aHR}$).
2. **Co-mutation & Interaction Analysis:** Investigate interaction terms between co-occurring mutations (e.g., dual TP53 / *EGFR* alterations) to assess whether concurrent mutations lower or increase mortality risk.
3. **Treatment-Stratified Subgroup Models:** Perform stratified survival analyses based on first-line therapeutic interventions (e.g., targeted therapy vs. platinum-based chemotherapy vs. immunotherapy) to isolate predictive vs. prognostic biomarkers.